In [1]:
import pandas as pd
import numpy as np
import pyfixest as pf

panel = pd.read_csv("../data/processed/panel.csv")

In [2]:
gdp = pd.read_csv("../data/raw/eurostat/nama_10_pc.tsv", sep="\t")
key = gdp.columns[0]
codes = gdp[key].str.split(",", expand=True)
codes.columns = ["freq", "unit", "na_item", "geo"]
gdp = pd.concat([codes, gdp.drop(columns=key)], axis=1)

gdp = gdp[(gdp["unit"] == "CP_PPS_EU27_2020_HAB") & (gdp["na_item"] == "B1GQ")]
gdp = gdp.melt(id_vars=["freq", "unit", "na_item", "geo"], var_name="year", value_name="raw")
gdp["year"] = gdp["year"].str.strip().astype(int)
gdp["gdp_pc"] = pd.to_numeric(gdp["raw"].str.extract(r"([\d.]+)")[0], errors="coerce")
gdp["flag"] = gdp["raw"].str.replace(r"[\d.:\s]", "", regex=True)

g25 = gdp[gdp["year"] == 2025][["geo", "gdp_pc", "flag"]]
g25[g25["geo"].isin(panel["geo"])].sort_values("gdp_pc")

,geo,gdp_pc,flag
2054,BG,28433.4,
2065,EL,29083.4,p
2089,SK,31648.4,
2071,HU,31815.6,p
2085,RO,32575.8,p
2083,PL,33859.2,
2084,PT,34022.2,p
2075,LT,36536.6,
2066,ES,38161.7,p
2056,CZ,38557.0,


In [3]:
panel = panel.merge(g25[["geo", "gdp_pc"]], on="geo", how="left")
panel["ln_gdp"] = np.log(panel["gdp_pc"])
panel["isco1"] = panel["isco2"].str[:3]
panel.to_csv("../data/processed/panel_gdp.csv", index=False)

d = panel[panel["main_sample"]].dropna(subset=["ln_s_may", "ln_e_2025"]).copy()
d["country_id"] = d["geo"].astype("category").cat.codes
d["gap"] = d["ln_s_may"] - d["ln_e_2025"]

groups = sorted(d["isco1"].unique())
slope_cols = []
for g in groups[1:]:
    d["gdp_x_" + g] = d["ln_gdp"] * (d["isco1"] == g)
    slope_cols.append("gdp_x_" + g)
slopes = " + ".join(slope_cols)

print(len(d), groups)
print(slopes)

482 ['OC1', 'OC2', 'OC3', 'OC4', 'OC5', 'OC7', 'OC8', 'OC9']
gdp_x_OC2 + gdp_x_OC3 + gdp_x_OC4 + gdp_x_OC5 + gdp_x_OC7 + gdp_x_OC8 + gdp_x_OC9


In [4]:
ols2 = pf.feols(f"ln_s_may ~ ln_e_2025 + {slopes} | geo + isco2", data=d, vcov={"CRV1": "country_id"})
ols2_vs1 = pf.feols(f"gap ~ ln_e_2025 + {slopes} | geo + isco2", data=d, vcov={"CRV1": "country_id"})
ols2.summary()
print(ols2.wildboottest(param="ln_e_2025", reps=9999, seed=42))
print(ols2_vs1.wildboottest(param="ln_e_2025", reps=9999, seed=42))

###

Estimation:  OLS
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |      0.030 |        0.038 |     0.809 |      0.431 | -0.050 |   0.110 |
| gdp_x_OC2     |     -0.089 |        0.153 |    -0.583 |      0.568 | -0.414 |   0.236 |
| gdp_x_OC3     |     -0.164 |        0.142 |    -1.158 |      0.265 | -0.466 |   0.138 |
| gdp_x_OC4     |     -0.070 |        0.158 |    -0.447 |      0.661 | -0.406 |   0.265 |
| gdp_x_OC5     |      0.026 |        0.172 |     0.153 |      0.881 | -0.341 |   0.393 |
| gdp_x_OC7     |     -0.083 |        0.258 |    -0.323 |      0.751 | -0.633 |   0.466 |
| gdp_x_OC8     |     -0.476 |        0.299 |    -1.593 |      0.132 | -1.112 |   0.161 |
| gdp_x_OC9     |      0.091 |        0.242 |     0.376 |      0

In [5]:
iv2 = pf.feols(f"ln_s_may ~ {slopes} | geo + isco2 | ln_e_2025 ~ ln_e_2015", data=d, vcov={"CRV1": "country_id"})
iv2.summary()
print(iv2._f_stat_1st_stage)

ppml2 = pf.fepois(f"s_may ~ ln_e_2025 + {slopes} | geo + isco2", data=d, vcov={"CRV1": "country_id"})
ppml2.summary()

###

Estimation:  IV
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| gdp_x_OC2     |     -0.090 |        0.157 |    -0.575 |      0.574 | -0.426 |   0.245 |
| gdp_x_OC3     |     -0.165 |        0.144 |    -1.145 |      0.270 | -0.471 |   0.142 |
| gdp_x_OC4     |     -0.075 |        0.163 |    -0.458 |      0.653 | -0.423 |   0.273 |
| gdp_x_OC5     |      0.015 |        0.177 |     0.082 |      0.936 | -0.362 |   0.391 |
| gdp_x_OC7     |     -0.110 |        0.255 |    -0.433 |      0.671 | -0.653 |   0.432 |
| gdp_x_OC8     |     -0.512 |        0.300 |    -1.703 |      0.109 | -1.152 |   0.129 |
| gdp_x_OC9     |      0.075 |        0.243 |     0.307 |      0.763 | -0.444 |   0.594 |
| ln_e_2025     |      0.009 |        0.041 |     0.219 |      0.

In [6]:
gdp[gdp["year"].isin([2024, 2025])][["geo", "year", "gdp_pc", "flag"]].to_csv("../data/processed/gdp.csv", index=False)